# Tutorial 3: Batch Workflow with SessionContext

**批量工作流 -- 使用 SessionContext 管理多棵树的模板生成和会话快照**

This notebook demonstrates how to use PyiTOL's `SessionContext` to manage batch
processing of multiple phylogenetic trees. We will:

1. Generate multiple synthetic trees in a directory
2. Process each tree through a template generation pipeline
3. Use `SessionContext` to track inputs, outputs, and generated files
4. Save session snapshots for reproducibility and replay
5. Demonstrate the session replay concept

本教程展示如何使用 PyiTOL 的 `SessionContext` 批量处理多棵系统发育树，
包括会话跟踪、快照保存和会话重放。

---

**Motivation / 动机:**

In real research projects, you often have dozens or hundreds of trees that need the same
annotation workflow. PyiTOL's `SessionContext` provides:
- Centralized tracking of all inputs and outputs
- Structured logging of each template generation step
- Snapshot saving for reproducibility (YAML format)
- Ability to reconstruct a session from a snapshot (replay)

## 1. Setup and Imports

导入所需的库。

In [ ]:
import random
import shutil
import tempfile
from pathlib import Path

import dendropy
import pandas as pd
import yaml

from pyitol.utils.session import SessionContext
from pyitol.templates.generator import (
    generate_color_strip_template,
    generate_heatmap_template,
    generate_simple_bar_template,
)
from pyitol.templates.presets.colorblind import get_colorblind_palette

## 2. Prepare Multiple Synthetic Trees

We create a directory with 3 synthetic trees, each representing a different gene.
Each tree has its own taxonomy CSV file.

创建包含 3 棵合成树的目录，每棵树代表不同的基因，各有对应的分类学 CSV 文件。

In [ ]:
random.seed(42)

# Create project structure
project_dir = Path(tempfile.mkdtemp(prefix="pyitol_batch_"))
trees_dir = project_dir / "trees"
metadata_dir = project_dir / "metadata"
output_dir = project_dir / "output"

trees_dir.mkdir()
metadata_dir.mkdir()
output_dir.mkdir()

# Phyla for synthetic taxonomy
phyla = ["Proteobacteria", "Firmicutes", "Actinobacteria", "Bacteroidetes"]
genera_by_phylum = {
    "Proteobacteria": ["Escherichia", "Pseudomonas", "Salmonella"],
    "Firmicutes": ["Bacillus", "Staphylococcus", "Streptococcus"],
    "Actinobacteria": ["Mycobacterium", "Corynebacterium", "Streptomyces"],
    "Bacteroidetes": ["Bacteroides", "Flavobacterium", "Cytophaga"],
}

gene_names = ["16S_rRNA", "rpoB", "gyrB"]
n_leaves_list = [30, 25, 35]

tree_paths = []
taxonomy_paths = []

for gene_name, n_leaves in zip(gene_names, n_leaves_list):
    # Generate tree
    tree = dendropy.Tree.purebirth_taxa_tree(num_leaves=n_leaves)
    for i, leaf in enumerate(tree.leaf_node_iter(), 1):
        leaf.taxon.label = f"{gene_name}_Seq_{i:03d}"

    tree_path = trees_dir / f"{gene_name}.nwk"
    tree.write_to_path(str(tree_path), schema="newick")
    tree_paths.append(tree_path)

    # Generate matching taxonomy
    records = []
    for leaf in tree.leaf_node_iter():
        phylum = random.choice(phyla)
        genus = random.choice(genera_by_phylum[phylum])
        records.append({
            "id": leaf.taxon.label,
            "Phylum": phylum,
            "Genus": genus,
            "Abundance": round(random.uniform(0.5, 20.0), 2),
            "GC_Content": round(random.uniform(35.0, 65.0), 1),
        })

    tax_df = pd.DataFrame(records)
    tax_path = metadata_dir / f"{gene_name}_taxonomy.csv"
    tax_df.to_csv(tax_path, index=False)
    taxonomy_paths.append(tax_path)

    print(f"{gene_name}: {n_leaves} leaves, tree={tree_path.name}, taxonomy={tax_path.name}")

print(f"\nProject directory: {project_dir}")

## 3. Process Each Tree with SessionContext

We create a `SessionContext` to track the entire batch workflow. The context records:
- Input files (tree, taxonomy)
- Output directory
- All generated template files
- Structured log entries

使用 `SessionContext` 跟踪整个批量工作流。

In [ ]:
# Create a session context for the batch run
session = SessionContext(
    tree_path=str(tree_paths[0]),  # primary tree
    taxonomy_path=str(taxonomy_paths[0]),
    id_column="id",
    output_dir=str(output_dir),
)
session.start()
session.record_command(["pyitol", "batch", "--trees", str(trees_dir), "--output", str(output_dir)])

print(f"Session started: {session.start_time}")
print(f"Output directory: {session.output_dir}")

In [ ]:
# Process each tree -- generate color-strip, heatmap, and bar templates
palette = get_colorblind_palette("tol_bright", n=4)
phylum_colors = dict(zip(phyla, palette))

all_generated_files = []

for tree_path, tax_path in zip(tree_paths, taxonomy_paths):
    gene = tree_path.stem
    print(f"\n--- Processing {gene} ---")

    # Record inputs for this iteration
    session.record_input(f"{gene}_tree", tree_path)
    session.record_input(f"{gene}_taxonomy", tax_path)

    # 1. Color-strip by Phylum
    strip_out = output_dir / f"{gene}_phylum_strip.txt"
    generate_color_strip_template(
        output_path=strip_out,
        taxonomy_path=tax_path,
        tree_path=tree_path,
        column="Phylum",
        colors=phylum_colors,
        label=f"{gene}_Phylum",
    )
    session.log_template("color_strip", strip_out, {"column": "Phylum", "gene": gene})
    session.register_output(f"{gene}_strip", strip_out)
    all_generated_files.append(strip_out)
    print(f"  Color-strip: {strip_out.name}")

    # 2. Heatmap
    heatmap_out = output_dir / f"{gene}_heatmap.txt"
    generate_heatmap_template(
        output_path=heatmap_out,
        taxonomy_path=tax_path,
        tree_path=tree_path,
        value_columns=["Abundance", "GC_Content"],
        color_gradient=["#d73027", "#ffffbf", "#4575b4"],
        label=f"{gene}_heatmap",
    )
    session.log_template("heatmap", heatmap_out, {"columns": ["Abundance", "GC_Content"], "gene": gene})
    session.register_output(f"{gene}_heatmap", heatmap_out)
    all_generated_files.append(heatmap_out)
    print(f"  Heatmap:     {heatmap_out.name}")

    # 3. Bar chart
    bar_out = output_dir / f"{gene}_bar.txt"
    generate_simple_bar_template(
        output_path=bar_out,
        taxonomy_path=tax_path,
        tree_path=tree_path,
        value_column="Abundance",
        label=f"{gene}_Abundance",
        bar_color="#3c5484",
    )
    session.log_template("bar", bar_out, {"column": "Abundance", "gene": gene})
    session.register_output(f"{gene}_bar", bar_out)
    all_generated_files.append(bar_out)
    print(f"  Bar chart:   {bar_out.name}")

session.finish()
print(f"\nBatch processing complete. Generated {len(all_generated_files)} template files.")

## 4. Review Session Summary

The `SessionContext` tracks all generated files and log entries.

查看会话摘要，了解所有生成的文件和日志条目。

In [ ]:
summary = session.get_summary()
print("Session Summary:")
print(f"  Tree:          {summary['tree']}")
print(f"  Taxonomy:      {summary['taxonomy']}")
print(f"  Output dir:    {summary['output_dir']}")
print(f"  Generated files: {len(summary['generated_files'])}")
for f in summary["generated_files"]:
    print(f"    - {Path(f).name}")

In [ ]:
# View the structured logs
logs = session.get_logs(limit=20)
print(f"Session logs ({len(logs)} entries):")
for log in logs:
    print(f"  [{log['timestamp']}] {log['action']}: {log['detail']}")

## 5. Save Session Snapshot

The session snapshot is a YAML file that captures the complete state of the workflow.
This enables reproducibility -- anyone can reconstruct the exact same workflow from
the snapshot.

会话快照是 YAML 文件，捕获工作流的完整状态，支持可重现性。

In [ ]:
# Save the session snapshot
snapshot_path = output_dir / "session_snapshot.yaml"
session.save_snapshot(
    output_path=snapshot_path,
    generated_files=all_generated_files,
)

print(f"Snapshot saved to: {snapshot_path}")
print(f"Snapshot size: {snapshot_path.stat().st_size} bytes")

In [ ]:
# Display the snapshot content
snapshot_data = yaml.safe_load(snapshot_path.read_text())
print("Snapshot structure:")
for key, value in snapshot_data.items():
    if isinstance(value, list):
        print(f"  {key}: [{len(value)} items]")
    elif isinstance(value, dict):
        print(f"  {key}: {{dict with {len(value)} keys}}")
    else:
        print(f"  {key}: {value}")

In [ ]:
# Show the inputs section
print("Inputs recorded in snapshot:")
for k, v in snapshot_data.get("inputs", {}).items():
    print(f"  {k}: {v}")

print("\nOutputs recorded in snapshot:")
outputs = snapshot_data.get("outputs", {})
for k, v in outputs.items():
    if k == "generated_files":
        print(f"  {k}: [{len(v)} files]")
    else:
        print(f"  {k}: {v}")

In [ ]:
# Show file checksums (for reproducibility verification)
files_meta = snapshot_data.get("files", [])
print(f"File metadata ({len(files_meta)} files):")
for f in files_meta:
    path_name = Path(f["path"]).name
    size = f.get("size", 0)
    checksum = f.get("checksum", "N/A")[:12]
    print(f"  {path_name:35s}  {size:>6d} bytes  md5={checksum}...")

## 6. Session Replay (Reconstruction)

One of the key features of `SessionContext` is the ability to reconstruct a session
from a saved snapshot. This is useful for:
- Reproducing someone else's analysis
- Debugging a failed workflow
- Auditing what was done in a previous run

`SessionContext` 的一个重要功能是从保存的快照重建会话，
这在重现分析、调试失败的工作流或审计之前的运行时非常有用。

In [ ]:
# Reconstruct a session from the snapshot
reconstructed = SessionContext.from_snapshot(snapshot_path)

print("Reconstructed session:")
print(f"  Tree path:      {reconstructed.tree_path}")
print(f"  Taxonomy path:  {reconstructed.taxonomy_path}")
print(f"  ID column:      {reconstructed.id_column}")
print(f"  Output dir:     {reconstructed.output_dir}")
print(f"  Start time:     {reconstructed.start_time}")
print(f"  Command args:   {reconstructed.command_line_args}")
print(f"  Input files:    {len(reconstructed.input_files)} entries")
print(f"  Output files:   {len(reconstructed.output_files)} entries")
print(f"  Generated files: {len(reconstructed.generated_files)} files")
print(f"  Logs:           {len(reconstructed._logs)} entries")

In [ ]:
# Verify the reconstructed session has the same generated files
original_files = set(str(Path(f)) for f in session.generated_files)
reconstructed_files = set(str(f) for f in reconstructed.generated_files)

print(f"Original files:      {len(original_files)}")
print(f"Reconstructed files: {len(reconstructed_files)}")
print(f"Match: {original_files == reconstructed_files}")

# Verify checksums are preserved
print(f"\nChecksums preserved: {len(reconstructed.file_checksums)} entries")

## 7. List All Generated Output Files

查看批量工作流生成的所有文件。

In [ ]:
print("All generated template files:")
print("-" * 60)
for f in sorted(output_dir.iterdir()):
    size = f.stat().st_size
    print(f"  {f.name:40s}  {size:>6d} bytes")

print("\nProject structure:")
for d in sorted(project_dir.rglob("*")):
    if d.is_file():
        rel = d.relative_to(project_dir)
        print(f"  {rel}")

## 8. Summary

In this tutorial we demonstrated PyiTOL's batch workflow capabilities:

1. **Multi-tree preparation**: Created a directory with 3 synthetic gene trees and matching taxonomy files
2. **Batch processing**: Generated color-strip, heatmap, and bar chart templates for each tree
3. **SessionContext tracking**: Recorded all inputs, outputs, and log entries in a structured context
4. **Session snapshots**: Saved the complete workflow state as a YAML file with file checksums
5. **Session replay**: Reconstructed a session from the snapshot and verified file integrity

**本教程总结：**

我们展示了 PyiTOL 的批量工作流功能，包括多棵树准备、批量模板生成、
SessionContext 跟踪、YAML 快照保存（含校验和），以及会话重放和完整性验证。

---

### Key Takeaways / 关键要点

- `SessionContext` provides a centralized way to track all workflow inputs and outputs
- `save_snapshot()` creates a YAML file with MD5 checksums for reproducibility
- `from_snapshot()` reconstructs the session, enabling exact workflow replay
- The `log_template()` method provides structured logging for each template generation step
- For real projects, combine this with `ITOLAPIClient` for automated upload-export-delete cycles

---

### Production Usage / 生产环境用法

In a real batch workflow with iTOL upload, you would add:

```python
from pyitol.api.client import ITOLAPIClient

client = ITOLAPIClient(api_key_file=".itolapi.key")
session.record_api_params({"api_key_file": ".itolapi.key", "format": "pdf"})

for tree_path, tax_path in zip(tree_paths, taxonomy_paths):
    # ... generate templates ...
    tree_id = client.upload(str(tree_path), template_files)
    session.log_task_upload(tree_id)
    # ... export, delete, etc.
```

In [ ]:
# Cleanup (optional)
# shutil.rmtree(project_dir)
print(f"Tutorial complete. Project files are in: {project_dir}")